# Tuesday · Lab 1 — Your First Neural Network

Today you'll teach a computer to read **handwritten digits** (0–9). This is the classic "hello world" of machine learning, and it gives you the exact skills you'll reuse **tomorrow** to recognize your own objects.

Every neural network follows the same plan:
1. **Data** — thousands of labeled examples.
2. **Model** — a network of simple units with adjustable knobs (*weights*).
3. **Training** — show it examples, measure how wrong it is, nudge the weights. Repeat.
4. **Test** — check it on digits it has never seen.

**How this lab works:** the numbered sections build and train the network. Scattered through them are **Your turn** boxes — small tasks where *you* change or write the code. Do them as you go; mentors are circulating and watching the chat. Aim to finish the sections by early afternoon — the last lab block moves to **Lab 2 (Experiments)**.

> **How to run this:** You're on our GPU server through **JupyterHub**, right in your browser — everything is already installed. Run each cell with **Shift+Enter**, top to bottom. When the next code cell prints `Training on: cuda`, you're using the GPU.

In [ ]:
# You're on our JupyterHub GPU server — torch, torchvision, and matplotlib are already installed.
# Only if you run this on your own laptop instead:
# !pip install torch torchvision matplotlib


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

torch.manual_seed(0)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Training on:", DEVICE)


### Your turn — warm-up check
The cell above should print `Training on: cuda`. If it says `cpu`, say so in the chat before you go on — you'd be training on the slow path.

## 1. The data: MNIST

MNIST is 70,000 small (28×28 pixel) grayscale images of handwritten digits, each labeled with the correct number. torchvision downloads it for us. We split it into a **training** set (to learn from) and a **test** set (to check honestly — the model never trains on these).

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),                 # image -> tensor, values 0..1
    transforms.Normalize((0.1307,), (0.3081,)),  # MNIST's mean/std
])

BATCH_SIZE = 64    # <-- how many images the network sees per update (Your turn A)

train_ds = datasets.MNIST("data", train=True, download=True, transform=transform)
test_ds = datasets.MNIST("data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=1000, shuffle=False)

print(f"{len(train_ds)} training images, {len(test_ds)} test images")
print(f"{len(train_loader)} batches per epoch (= {len(train_ds)} ÷ {BATCH_SIZE})")


### Your turn A — resize the batches
A **batch** is the group of images the network looks at before each tune-up. Right now `BATCH_SIZE = 64`. Change it to `256` in the cell above and re-run this section. How many batches per epoch now? Later, when you train, notice whether it feels faster or slower and whether the final accuracy changes.

> _your notes (double-click to edit):_

## 2. Look at the data first

Each image is 28×28 = 784 numbers (pixel brightnesses). Here are a few, with their labels. **Always eyeball your data before training** — it's the fastest way to catch problems.

In [ ]:
import matplotlib.pyplot as plt

imgs, labels = next(iter(train_loader))
plt.figure(figsize=(9, 3))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(imgs[i][0], cmap="gray")
    plt.title(f"label: {labels[i].item()}")
    plt.axis("off")
plt.tight_layout(); plt.show()


### Your turn B — look closer
Below is a working copy that shows **15** digits in a 3×5 grid. Run it. Then try the `# TODO`: show only the images whose label is `7`. (Hint: build a list of indices where `labels == 7`, then plot those.)

In [ ]:
imgs, labels = next(iter(train_loader))
plt.figure(figsize=(9, 5))
for i in range(15):
    plt.subplot(3, 5, i + 1)
    plt.imshow(imgs[i][0], cmap="gray")
    plt.title(f"{labels[i].item()}")
    plt.axis("off")
plt.tight_layout(); plt.show()

# TODO: show only the 7s. Start here:


## 3. Build the network

A simple network of fully-connected layers — exactly the kind in the 3Blue1Brown videos:

- **Flatten** the 28×28 image into a list of 784 numbers.
- A **hidden layer** of 128 units, then 64 units, each followed by a **ReLU** (keeps positives, zeros out negatives — this is the non-linear "bend" that lets the network learn curvy patterns).
- A final layer with **10 outputs**, one score per digit. The highest score is the guess.

Every arrow between units has a **weight** — a knob training will adjust. This little network has about 100,000 of them.

In [ ]:
class DigitNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 128), nn.ReLU(),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, 10),
        )

    def forward(self, x):
        return self.net(x)

model = DigitNet().to(DEVICE)
print(model)
print("Total weights:", sum(p.numel() for p in model.parameters()))


### Your turn C — ask an *untrained* network
The network exists but hasn't learned anything — its knobs are random. Run one image through it and look at the guess. With 10 possible digits, random guessing is about **10%** confident-correct. This is your "before" picture; keep it in mind when you see the "after".

In [ ]:
img, true_label = test_ds[0]
probs = F.softmax(model(img.unsqueeze(0).to(DEVICE))[0], dim=0)
guess = int(probs.argmax())
print(f"Untrained network guesses: {guess}  (it's actually {true_label})")
print(f"Confidence: {probs[guess].item():.0%}  — basically a coin-flip across 10 options")


## 4. Loss and optimizer

- The **loss** measures how wrong the network is (`CrossEntropyLoss` is standard for "pick one of N classes").
- The **optimizer** does the nudging. `Adam` reads the loss and adjusts every weight a little to reduce it.

The `run_epoch` function below is the heart of training — and it's the **same function** you'll use tomorrow for your object recognizer.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


def run_epoch(loader, train):
    model.train() if train else model.eval()
    total, correct, loss_sum = 0, 0, 0.0
    torch.set_grad_enabled(train)
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        if train:
            optimizer.zero_grad(); loss.backward(); optimizer.step()
        loss_sum += loss.item() * imgs.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += imgs.size(0)
    return loss_sum / total, correct / total


## 5. Train

Each **epoch** is one full pass over the training data. Watch the accuracy climb. A tiny network like this usually reaches ~97% on unseen test digits in just a few epochs.

In [ ]:
EPOCHS = 5    # <-- Your turn D changes this
for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(train_loader, train=True)
    te_loss, te_acc = run_epoch(test_loader, train=False)
    print(f"epoch {epoch}  train acc {tr_acc:5.1%}   test acc {te_acc:5.1%}")


### Your turn D — how many epochs?
Change `EPOCHS` above and re-run this section (you'll want to re-run section 3 first to start from a fresh, untrained network each time). Fill in the table:

| EPOCHS | final test acc |
|---|---|
| 1 |  |
| 3 |  |
| 5 |  |
| 10 |  |

Where does accuracy basically stop improving? That's your clue that you've trained "enough" — more epochs after that mostly waste time (and can start to overfit).

## 6. See what it learned — and where it fails

The mistakes are the interesting part. Below are digits the network got **wrong**. Many are genuinely messy — a good reminder that models are only as clear as their data.

In [ ]:
imgs, labels = next(iter(test_loader))
with torch.no_grad():
    preds = model(imgs.to(DEVICE)).argmax(1).cpu()

wrong = (preds != labels).nonzero(as_tuple=True)[0][:10]
plt.figure(figsize=(9, 3))
for i, idx in enumerate(wrong):
    plt.subplot(2, 5, i + 1)
    plt.imshow(imgs[idx][0], cmap="gray")
    plt.title(f"said {preds[idx].item()}, was {labels[idx].item()}")
    plt.axis("off")
plt.tight_layout(); plt.show()


### Your turn E — accuracy by hand
The test loader just handed you a batch of 1000 images. Count how many the network got **wrong** in this batch, and turn that into an accuracy. Does your number match the `test acc` printed during training?

In [ ]:
n_wrong = (preds != labels).sum().item()
n_total = len(labels)
print(f"{n_wrong} wrong out of {n_total}")
# TODO: print the accuracy as a percentage. 


## 7. Ask it about one digit

`predict` returns the guessed digit and how confident the network is — the same idea you'll use tomorrow to ask "is this the target object?"

In [ ]:
def predict(image_tensor):
    model.eval()
    with torch.no_grad():
        probs = F.softmax(model(image_tensor.unsqueeze(0).to(DEVICE))[0], dim=0).cpu()
    digit = int(probs.argmax())
    return digit, float(probs[digit])

img, true_label = test_ds[0]
digit, conf = predict(img)
print(f"The network sees a {digit} ({conf:.0%} sure). It's actually a {true_label}.")


### Your turn F — find a shaky guess
Most digits get a confident (~99%) answer. Hunt for the borderline ones. Below, loop over the first 200 test images and print any where the network is **less than 60% sure**. These wobbly cases are exactly what a confidence **threshold** handles on Friday: the arm only acts when it's sure enough.

In [ ]:
low_conf = []
for i in range(200):
    img, true_label = test_ds[i]
    digit, conf = predict(img)
    if conf < 0.60:
        low_conf.append((i, digit, true_label, conf))
        print(f"image {i:3d}: said {digit} ({conf:.0%}), was {true_label}")
print(f"\n{len(low_conf)} shaky guesses out of 200")
# TODO: show one of these images with plt.imshow to see WHY it's confusing.


## 8. Save it (optional)

```python
torch.save(model.state_dict(), "digitnet.pt")
```

### What just happened (the big picture)

You gave a network **examples**, it measured its **error**, and an optimizer **adjusted its weights** thousands of times until it got good. Tomorrow is the *exact same recipe* — the only changes are: your objects instead of digits, and a head start from a model already trained on millions of images (transfer learning) so you need far fewer examples.

## Finished the sections? → Lab 3

If your network trains past ~97% and you've done the **Your turn** boxes, open **Lab 3 (Experiments)** — resize the brain, add layers, build a CNN that breaks 99%, and hunt down the digit it confuses most. Your instructor will post the Lab 2 link.

Two quick ones you can also try right here first:
1. In section 3, change the hidden layers to `256` then `128`. More accurate? Slower?
2. Insert another `nn.Linear` + `nn.ReLU`. Does it help — or start to overfit?